# Example 10.5 / 1: Penalized unmet demand

Companion notebook for **10.5 Soft Constraints and Penalties** in the AMPL lab Documentation, section "Penalized unmet demand". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Penalized unmet demand

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param demand >= 0;  # units required
param capacity >= 0;  # maximum units produced
param unit_cost >= 0;  # cost per unit produced
param shortage_penalty >= 0;  # penalty per unit of unmet demand

var Produce >= 0, <= capacity;  # units produced
var Shortage >= 0;  # unmet demand; slack of the soft constraint

minimize TotalCost:  # production cost plus shortage penalty
    unit_cost * Produce + shortage_penalty * Shortage;

subject to DemandBalance:  # soft demand: shortage closes any gap
    Produce + Shortage = demand;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["demand"] = 120
ampl.param["capacity"] = 100
ampl.param["unit_cost"] = 4
ampl.param["shortage_penalty"] = 30

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Produce"].value(), ampl.var["Shortage"].value())

HiGHS 1.14.0: optimal solution; objective 1000
0 simplex iterations
0 barrier iterations
solved
100.0 20.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
param demand := 120;
param capacity := 100;
param unit_cost := 4;
param shortage_penalty := 30;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Produce, Shortage, TotalCost;

HiGHS 1.14.0: optimal solution; objective 1000
0 simplex iterations
0 barrier iterations
Produce = 100
Shortage = 20
TotalCost = 1000

